# Silver Layer — ICD Diagnosis Dictionary (`d_icd_diagnoses`)

## Source -> Target

`<source_schema>.d_icd_diagnoses` -> `<target_schema>.d_icd_diagnoses`

Defaults: `source_schema = cdr_mimiciv.bronze`, `target_schema = cdr_mimiciv.silver`. The target table is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

## Grain

One row per ICD diagnosis code and ICD version (`icd_code`, `icd_version`), as documented for MIMIC-IV `hosp.d_icd_diagnoses` (both columns NOT NULL; `icd_version` is 9 or 10; `long_title` is the description). `diagnoses_icd` links to this table on `icd_code` and `icd_version`.

## Silver objective

Provide a reusable, standardized ICD-9/ICD-10 diagnosis reference dictionary while preserving every source row, the original code values and the original descriptions.

## Engineering principles

- Preserve all source rows and original ICD codes; no decimal insertion or code mapping.
- Only deterministic type casting and whitespace trimming.
- Validate key completeness, key uniqueness, ICD versions and descriptions; fail publication rather than silently correct.
- No imputation, deduplication or analytical features (those belong in Gold).

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure source/target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

In [0]:
# Configure source and target tables
TABLE = "d_icd_diagnoses"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "icd_code": "string",
    "icd_version": "int",
    "long_title": "string",
}

## 1. Bronze source inspection

Read the original source table from the Bronze layer.

Inspect the available records, column names and datatypes to understand the source structure before applying transformations.

No source records are modified during this stage.

In [0]:
# Read the source Bronze table
bronze_df = spark.table(SOURCE_TABLE)

display(bronze_df.limit(20))

icd_code,icd_version,long_title
0010,9,Cholera due to vibrio cholerae
0011,9,Cholera due to vibrio cholerae el tor
0019,9,"Cholera, unspecified"
0020,9,Typhoid fever
0021,9,Paratyphoid fever A
0022,9,Paratyphoid fever B
0023,9,Paratyphoid fever C
0029,9,"Paratyphoid fever, unspecified"
0030,9,Salmonella gastroenteritis
0031,9,Salmonella septicemia


In [0]:
# Inspect source datatypes
bronze_df.printSchema()

root
 |-- icd_code: string (nullable = true)
 |-- icd_version: integer (nullable = true)
 |-- long_title: string (nullable = true)



## 2. Source data profiling

Evaluate source data quality before applying transformations.

The profiling stage checks:

- Total number of records.
- Missing values in source columns.
- Duplicate records based on the expected business grain.
- Unexpected or rescued source data.
- Validity of clinical coding systems and identifiers.

Profiling results determine which Silver transformations are necessary. Records are not automatically removed or corrected without validated rules.

In [0]:
# Count source records
print("Bronze rows:", bronze_df.count())

Bronze rows: 112107


In [0]:
# Count null values in every source column
null_profile = bronze_df.select(
    *[
        F.sum(
            F.when(F.col(f"`{c}`").isNull(), 1).otherwise(0)
        ).alias(c)
        for c in bronze_df.columns
    ]
)

display(null_profile)

icd_code,icd_version,long_title
0,0,0


In [0]:
# Check records containing rescued source data (column may not exist in redesigned Bronze)
if "_rescued_data" in bronze_df.columns:
    rescued_count = bronze_df.filter(
        F.col("_rescued_data").isNotNull()
    ).count()
    print("Rows with rescued data:", rescued_count)
else:
    print("No rescued-data column found")

No rescued-data column found


In [0]:
# Identify duplicate ICD code-version combinations
duplicate_records = (
    bronze_df
    .groupBy("icd_code", "icd_version")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicate_records)

icd_code,icd_version,count


## 3. Silver transformations

Transform the Bronze source into a reusable, standardized Silver entity.

Apply only justified, deterministic transformations, including:

- Appropriate datatype casting.
- Safe standardization of identifiers and text fields.
- Preservation of original clinical codes and meanings.
- Preservation of source relationships and record granularity.
- Separation of ingestion metadata from clinically relevant fields.

Do not impute missing clinical information, fabricate records or apply Gold-level analytical aggregations.

In [0]:
# Create the Silver ICD diagnosis dictionary
silver_df = (
    bronze_df
    .select(
        F.trim(F.col("icd_code").cast("string")).alias("icd_code"),
        F.col("icd_version").cast("int").alias("icd_version"),
        F.trim(F.col("long_title").cast("string")).alias("long_title")
    )
)

# Preview standardized Silver records
display(silver_df.limit(20))

# Inspect Silver schema
silver_df.printSchema()

icd_code,icd_version,long_title
0010,9,Cholera due to vibrio cholerae
0011,9,Cholera due to vibrio cholerae el tor
0019,9,"Cholera, unspecified"
0020,9,Typhoid fever
0021,9,Paratyphoid fever A
0022,9,Paratyphoid fever B
0023,9,Paratyphoid fever C
0029,9,"Paratyphoid fever, unspecified"
0030,9,Salmonella gastroenteritis
0031,9,Salmonella septicemia


root
 |-- icd_code: string (nullable = true)
 |-- icd_version: integer (nullable = true)
 |-- long_title: string (nullable = true)



In [0]:
# Identify duplicate ICD code-version combinations
duplicate_codes = (
    silver_df
    .groupBy("icd_code", "icd_version")
    .count()
    .filter(F.col("count") > 1)
)

# Display duplicate validation results
print("Duplicate ICD combinations:", duplicate_codes.count())
display(duplicate_codes)

Duplicate ICD combinations: 0


icd_code,icd_version,count


## 5. Silver Data Quality Validation

Evaluate the completeness and validity of the standardized diagnosis dictionary.

Checks performed:
- Missing or blank ICD codes.
- Missing ICD versions.
- Invalid ICD versions outside ICD-9 and ICD-10.
- Missing or blank diagnosis descriptions.

No missing clinical information is fabricated or automatically corrected.

In [0]:
# Validate missing and invalid values in Silver ICD diagnosis columns
icd_quality = silver_df.select(
    F.sum(
        F.when(
            F.col("icd_code").isNull() |
            (F.col("icd_code") == ""), 1
        ).otherwise(0)
    ).alias("missing_icd_codes"),

    F.sum(
        F.when(
            F.col("icd_version").isNull(), 1
        ).otherwise(0)
    ).alias("null_icd_versions"),

    F.sum(
        F.when(
            F.col("icd_version").isNotNull() &
            (~F.col("icd_version").isin(9, 10)), 1
        ).otherwise(0)
    ).alias("invalid_icd_versions"),

    F.sum(
        F.when(
            F.col("long_title").isNull() |
            (F.col("long_title") == ""), 1
        ).otherwise(0)
    ).alias("missing_descriptions")
)

# Display Silver quality results
display(icd_quality)

missing_icd_codes,null_icd_versions,invalid_icd_versions,missing_descriptions
0,0,0,0


## 6. Bronze-to-Silver Record Reconciliation

Compare Bronze and Silver record counts.

Since this reference-table transformation does not intentionally remove records, the total number of records should remain unchanged.

Any unexpected difference prevents publication.

In [0]:
# Compare Bronze and Silver record counts
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

# Stop processing if the record counts differ
assert bronze_count == silver_count, "Unexpected row-count change"

Bronze rows: 112107
Silver rows: 112107


In [0]:
# Check missing values in Silver ICD diagnosis columns
null_check = silver_df.select(
    *[
        F.sum(
            F.when(F.col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in [
            "icd_code",
            "icd_version",
            "long_title"
        ]
    ]
)

display(null_check)

icd_code,icd_version,long_title
0,0,0


## 7. Final Silver Quality Gate

Validate critical conditions before publishing the Silver table.

Publication requirements:
- No duplicate ICD code-version combinations.
- No missing or blank ICD codes.
- No missing ICD versions.
- No invalid ICD versions.
- No missing or blank diagnosis descriptions.
- Bronze and Silver record counts match.

Failed checks stop publication and require investigation.

In [0]:
# Collect data quality metrics
quality = icd_quality.first()

# Count duplicate ICD combinations
duplicate_count = duplicate_codes.count()

# Validate unique diagnosis identifiers
assert duplicate_count == 0, "Duplicate ICD code-version combinations found"

# Validate required ICD codes
assert quality["missing_icd_codes"] == 0, "Missing ICD codes found"

# Validate ICD versions
assert quality["null_icd_versions"] == 0, "Missing ICD versions found"
assert quality["invalid_icd_versions"] == 0, "Invalid ICD versions found"

# Validate diagnosis descriptions
assert quality["missing_descriptions"] == 0, "Missing diagnosis descriptions found"

# Confirm record-count preservation
assert bronze_count == silver_count, "Bronze and Silver counts do not match"

print("All Silver quality checks passed.")

All Silver quality checks passed.


## 8. Publish Silver ICD Diagnosis Dictionary

Write the validated Silver dataset as a managed Delta table in Unity Catalog using `CREATE OR REPLACE` semantics.

Target: `<target_schema>.d_icd_diagnoses` (default `cdr_mimiciv.silver.d_icd_diagnoses`). Downstream Gold depends on this table.

The Silver table provides a reusable diagnosis reference entity for downstream clinical analytics and Gold-layer development.

In [0]:
# Publish the validated ICD diagnosis dictionary to Silver (create or replace)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

# Confirm successful publication
print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.d_icd_diagnoses


## 9. Published Silver Verification

Read the published Delta table and verify:

- The table is accessible through Unity Catalog.
- The published record count matches the validated Silver dataset.
- The published schema contains the expected columns.
- The data remains available for downstream Gold transformations.

In [0]:
# Read the published Silver table
published_df = spark.table(TARGET_TABLE)

# Verify published record count
published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

# Confirm published data completeness
assert published_count == bronze_count, "Published record count mismatch"

# Confirm the published schema matches the contract (ordered)
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

# Confirm key completeness and uniqueness on the published table
published_keys = published_df.select(
    F.count("*").alias("rows"),
    F.sum(
        (F.col("icd_code").isNull() | F.col("icd_version").isNull()).cast("int")
    ).alias("null_keys"),
    F.countDistinct("icd_code", "icd_version").alias("distinct_keys")
).first()

assert published_keys["null_keys"] == 0, "Published key columns contain NULLs"
assert published_keys["distinct_keys"] == published_count, "Published keys are not unique"

display(published_df.limit(20))

print("Silver ICD diagnosis dictionary verified successfully.")

Bronze rows: 112107
Silver rows: 112107
Published rows: 112107


icd_code,icd_version,long_title
0010,9,Cholera due to vibrio cholerae
0011,9,Cholera due to vibrio cholerae el tor
0019,9,"Cholera, unspecified"
0020,9,Typhoid fever
0021,9,Paratyphoid fever A
0022,9,Paratyphoid fever B
0023,9,Paratyphoid fever C
0029,9,"Paratyphoid fever, unspecified"
0030,9,Salmonella gastroenteritis
0031,9,Salmonella septicemia


Silver ICD diagnosis dictionary verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")